# AutoML Quickstart: AutoGluon and PyCaret

Use AutoML as a budgeted baseline while preserving a held-out test set and externalizing artifacts.

- **Study time:** 30 minutes to review; runtime depends on the chosen time budget
- **Prerequisites:** tabular train/test workflow and metric selection
- **Mode:** `heavy`
- **Data policy:** no downloads; synthetic tabular data; model artifacts resolve to the external data root; AutoGluon and PyCaret use separate external environments; execution is opt-in with RUN_AUTOML=1
- **Provenance:** consolidated from the legacy AutoGluon and PyCaret teaching notebooks; errorful exploratory cells removed

Output convention: every retained textual result begins with a label that identifies the operation that produced it.
Annotation convention: comments explain intent, shape changes, invariants, subtle API behavior, or configuration side effects; obvious Python syntax is left uncommented.


In [ ]:
import importlib.util
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from datacoding.config import external_path

rng = np.random.default_rng(81)  # Reproducible local generator without global RNG side effects.

n_rows = 600
data = pd.DataFrame(
    {
        "age": rng.integers(18, 75, size=n_rows),
        "income": rng.normal(65_000, 18_000, size=n_rows),
        "channel": rng.choice(["web", "app", "store"], size=n_rows),
    }
)
logit = (
    -3.0 + 0.00004 * data["income"] + 0.018 * data["age"] + (data["channel"] == "app") * 0.4
)  # Controlled synthetic signal.
probability = 1 / (1 + np.exp(-logit))
data["label"] = (rng.random(n_rows) < probability).astype(int)
train_data, test_data = train_test_split(
    data,
    test_size=0.25,
    random_state=42,
    stratify=data["label"],  # Preserve class balance in the untouched test set.
)

# Check availability without importing either heavy stack.
has_autogluon = importlib.util.find_spec("autogluon") is not None
has_pycaret = importlib.util.find_spec("pycaret") is not None
run_automl = os.environ.get("RUN_AUTOML") == "1"  # Explicit opt-in prevents accidental heavy runs.

print("Dataset | train/test shapes", (train_data.shape, test_data.shape))
print(
    "Optional stack | availability",
    {"AutoGluon": has_autogluon, "PyCaret": has_pycaret, "RUN_AUTOML": run_automl},
)

## 1. AutoGluon with an explicit time budget and external model path


In [ ]:
def run_autogluon(train_frame, test_frame, time_limit=120):
    from autogluon.tabular import TabularPredictor

    # Keep generated artifacts outside the Obsidian vault.
    model_path = external_path("models", "autogluon_tabular_demo")
    predictor = TabularPredictor(
        label="label",
        eval_metric="f1",  # Match model selection to the stated classification objective.
        path=str(model_path),
    )
    predictor.fit(
        train_data=train_frame,
        time_limit=time_limit,  # Make compute budget part of the experiment contract.
        presets="medium_quality",
    )
    held_out_metrics = predictor.evaluate(test_frame)
    # Reporting only: do not reselect a winner from test results.
    held_out_leaderboard = predictor.leaderboard(test_frame)
    return predictor, held_out_metrics, held_out_leaderboard


if run_automl and has_autogluon:
    autogluon_predictor, autogluon_score, autogluon_leaderboard = run_autogluon(
        train_data, test_data
    )
    print("AutoGluon | held-out metrics", autogluon_score)
    print("AutoGluon | leaderboard head", autogluon_leaderboard.head())
else:
    print(
        "AutoGluon | execution status",
        "skipped; activate the autogluon environment and set RUN_AUTOML=1",
    )

## 2. PyCaret object-oriented experiment API


In [ ]:
def run_pycaret(train_frame, test_frame):
    from pycaret.classification import ClassificationExperiment

    experiment = ClassificationExperiment()
    experiment.setup(
        data=train_frame,
        target="label",
        session_id=42,  # Reproducible setup and cross-validation.
        html=False,
        verbose=False,
    )
    # Select by CV while favoring the quick-baseline model set.
    best_model = experiment.compare_models(turbo=True)
    finalized_model = experiment.finalize_model(best_model)  # Refit the winner on all setup rows.
    predictions = experiment.predict_model(
        finalized_model, data=test_frame
    )  # Evaluate once on held-out rows.
    return experiment, finalized_model, predictions


if run_automl and has_pycaret:
    pycaret_experiment, pycaret_model, pycaret_predictions = run_pycaret(train_data, test_data)
    print("PyCaret | selected model", pycaret_model)
    print("PyCaret | held-out prediction columns", pycaret_predictions.columns.tolist())
else:
    print(
        "PyCaret | execution status",
        "skipped; activate the pycaret environment and set RUN_AUTOML=1",
    )

## 3. Fair-comparison checklist

1. Use the same train/test definition as the manual baseline.
2. Select the metric before comparing models.
3. Record the time and compute budget.
4. Keep the test set out of model selection.
5. Inspect leaderboard failures and inference latency.
6. Store models and logs outside the vault.
